# SRA Native LLM Integration PoC (TinyLlama)

このノートブックは、SRAのルーターを既存のLLM（ここではLlamaアーキテクチャの軽量版であるTinyLlama）の次元数に合わせてネイティブに統合する概念実証（PoC）です。
Google Colab (無料枠のT4 GPU等) で動作するように設計されています。

In [ ]:
# Google Colab等のための環境構築
!pip install -q transformers torch numpy

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM

# デバイスの設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# TinyLlamaのロード (Llamaアーキテクチャの軽量版, 約1.1Bパラメータ)
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_llm = AutoModelForCausalLM.from_pretrained(model_id).to(device)

# LLMの重みを凍結（SRAルーターのみを学習させるため）
for param in base_llm.parameters():
    param.requires_grad = False

# TinyLlamaの隠れ層の次元数を取得 (通常は2048次元)
hidden_size = base_llm.config.hidden_size
print(f"LLM Hidden Size (SRA Native Dimension): {hidden_size}")

## SRAルーターと外部モジュール（シナプス）の定義

SRAのベースシステムを「LLMと同じ次元数」で構築します。これによりプロジェクション層が不要になります。

In [ ]:
class SRARouter(nn.Module):
    def __init__(self, d_model, num_experts=2):
        super().__init__()
        # 極めて軽量なルーティング層（LLM本体の学習に比べてコストはほぼゼロ）
        self.gate = nn.Linear(d_model, num_experts)
        
    def forward(self, x):
        # x shape: [batch, seq_len, d_model]
        routing_logits = self.gate(x)
        routing_probs = torch.softmax(routing_logits, dim=-1)
        return routing_probs

class DummyCalculatorSynapse(nn.Module):
    """
    非学習の外部モジュール（計算機など）のモック。
    ネイティブ統合のため、LLMと同じd_modelを受け取り、d_modelを返す。
    """
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        # 実際にはここに計算機へテキストを渡し、結果をベクトル化する処理が入る
        
    def forward(self, x):
        # ここではモックとして、単にテンソルを0.5倍して返す（何らかの処理の代わり）
        print("  [Calculator Synapse] Activated for calculation!")
        return x * 0.5

## ネイティブ統合フォワードパスの検証

入力テキストをLLMのトークナイザーでベクトル化し、ルーターで「LLM本体」と「計算機」に振り分けるシミュレーションを行います。

In [ ]:
# 1. SRAシステムの初期化
router = SRARouter(d_model=hidden_size, num_experts=2).to(device=device, dtype=base_llm.dtype)
calculator_synapse = DummyCalculatorSynapse(d_model=hidden_size).to(device=device, dtype=base_llm.dtype)

# 2. 入力データの準備
text = "What is the result of 15 * 24 ?"
inputs = tokenizer(text, return_tensors="pt").to(device)

print(f"Input Text: '{text}'")
print(f"Token IDs shape: {inputs.input_ids.shape}")

# 3. LLMのEmbedding層でネイティブなベクトルに変換
with torch.no_grad():
    # TinyLlamaのEmbedding層を通す
    embeddings = base_llm.model.embed_tokens(inputs.input_ids)
print(f"Embeddings shape (Native): {embeddings.shape}")

# 4. ルーティングの実行
# プロジェクション層なしで直接ルーターに渡せる！
routing_probs = router(embeddings)
print(f"Routing Probabilities shape: {routing_probs.shape}")

# 5. ルーティング結果に基づく処理の分岐（シミュレーション）
# トークンごとに、LLM(Expert 0)に送るか、Calculator(Expert 1)に送るかを決める
# ここでは簡略化のため、最後のトークン（'?'）のルーティング確率を見る
last_token_probs = routing_probs[0, -1, :]
print(f"Routing Probs for last token: LLM={last_token_probs[0]:.4f}, Calc={last_token_probs[1]:.4f}")

# 実際のSRAでは、ここで確率に応じて処理を分岐・結合します
print("\n--- Routing Execution ---")
if last_token_probs[1] > 0.5:
    # 計算機にルーティングされた場合（プロジェクションなしで直接渡せる！）
    out_tensor = calculator_synapse(embeddings)
else:
    # LLMにルーティングされた場合
    print("  [LLM Synapse] Processing via Frozen LLM Layers...")
    # 本来はTransformerブロックを通す
    out_tensor = embeddings 

print(f"Output Tensor shape: {out_tensor.shape}")
print("\n✅ ネイティブ統合（次元変換なしのシームレスな接続）の検証完了！")

## ルーターの学習（Router Training）

SRAの真価である「極めて軽量な学習」を実証します。
LLM本体の重みは凍結（Frozen）したまま、ルーター（わずか `d_model x 2` のパラメータ）だけを学習させます。
計算問題ならCalculatorへ、それ以外ならLLMへルーティングするように学習させます。

In [ ]:
import torch.optim as optim

# 1. トイデータセットの準備
train_data = [
    # 計算問題 (ラベル: 1 -> Calculator)
    ("What is 5 + 5?", 1),
    ("Calculate 12 * 3", 1),
    ("15 divided by 3 is", 1),
    ("Give me the result of 100 - 42", 1),
    
    # 日常会話/一般知識 (ラベル: 0 -> LLM)
    ("Hello, how are you?", 0),
    ("Tell me a story about a brave knight.", 0),
    ("What is the capital of France?", 0),
    ("Write a poem about the sea.", 0)
]

# 2. 損失関数とオプティマイザの定義
criterion = nn.CrossEntropyLoss()
# ★ルーターの重み（router.parameters()）だけを最適化対象にする
optimizer = optim.Adam(router.parameters(), lr=0.01)

print("Training setup complete. Ready to train!")

In [ ]:
# 3. 学習ループ
epochs = 30

print("Starting training...")
for epoch in range(epochs):
    total_loss = 0.0
    
    for text, label in train_data:
        optimizer.zero_grad()
        
        # 入力のベクトル化
        inputs = tokenizer(text, return_tensors="pt").to(device)
        with torch.no_grad():
            # LLM層は逆伝播しない（Frozen）
            embeddings = base_llm.model.embed_tokens(inputs.input_ids)
            
        # ルーターでロジットを取得（Softmax適用前の生の値）
        # .gate は直接Linear層なので、ロジットを返す
        routing_logits = router.gate(embeddings) # shape: [1, seq_len, 2]
        
        # シーケンス全体の平均を使って判断する
        mean_logits = routing_logits.mean(dim=1) # shape: [1, 2]
        
        # 正解ラベルの準備
        target = torch.tensor([label]).to(device)
        
        # 損失の計算と重みの更新
        loss = criterion(mean_logits, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {total_loss / len(train_data):.4f}")

print("Training finished! Router is now smart.")

## 学習後の推論検証

学習前は50:50の確率だったルーターが、入力テキストの意味を理解して正しいシナプス（モジュール）にルーティングできるようになったかを確認します。

In [ ]:
def test_routing(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        embeddings = base_llm.model.embed_tokens(inputs.input_ids)
        routing_probs = router(embeddings)
        
        # 学習時と同様にシーケンス平均の確率を見る
        mean_probs = routing_probs.mean(dim=1)[0]
        
    print(f"Input: '{text}'")
    print(f" -> Routing: LLM={mean_probs[0]:.4f} | Calc={mean_probs[1]:.4f}")
    
    if mean_probs[1] > 0.5:
        print("    => 🧮 Routed to Calculator Synapse!")
    else:
        print("    => 🧠 Routed to Frozen LLM Synapse!")
    print("-" * 40)

# テスト実行
print("\n=== Post-Training Routing Test ===\n")

# 1. 算数の問題（未学習のテキスト）
test_routing("What is the result of 15 * 24 ?")
test_routing("Can you compute 99 / 3 for me?")

# 2. 一般会話（未学習のテキスト）
test_routing("What is the meaning of life?")
test_routing("Explain quantum physics in simple terms.")